In [1]:
# Fix Windows symlink issue
import os
os.environ['HF_HUB_DISABLE_SYMLINKS'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
print("âœ… Disabled HuggingFace symlinks for Windows compatibility")

âœ… Disabled HuggingFace symlinks for Windows compatibility


# RAG on ASQA Long-Form QA Dataset

This notebook implements and evaluates a black-box RAG pipeline on the ASQA dataset:
1. **Normal RAG**: Standard retrieve-and-generate pipeline
2. **Black-box Answer Scoring**: Score generated answers against ground truth (no context filtering)
3. **Answer Reward Filter**: Generate-then-score with retry loop for better answers

ASQA focuses on ambiguous questions requiring long-form answers (50-200 words).

## 1. Setup and Imports

In [2]:
import sys
sys.path.append('..')

import os
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm
import dotenv

# Load environment variables
dotenv.load_dotenv()

# Import RAG system components
from src.config import RAGConfig
from src.rag_system import RAGSystem
from src.data.loader import ASQALoader

# Black-box answer scoring (scores answers against ground truth, no context filtering)
from src.filtering.llm_filter import AnswerFilter
from src.filtering.reward_filter import AnswerRewardFilter, AnswerRewardComputer
from src.filtering.metrics import AnswerMetricBundle
from src.filtering.weight_fitting import WeightFitter, WeightBank
from src.filtering.data_models import ANSWER_WEIGHT_PRIORS

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\notebooks\..\src\filtering\llm_filter.py:39: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 2. Load ASQA Dataset

In [3]:
# Initialize ASQA loader
loader = ASQALoader()

# Load data
train_path = "../data/asqa/train.csv"
dev_path = "../data/asqa/dev.csv"

df_train, df_dev = loader.load_data(train_path, dev_path)

# Display statistics
stats = loader.get_statistics()
print("\nDataset Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

Loading ASQA data...
Loaded 4353 training samples
Loaded 948 dev samples

Answer length statistics (words):
  Mean: 73.3
  Median: 68.0
  Min: 10
  Max: 491

Dataset Statistics:
  train_samples: 4353
  train_avg_docs_per_question: 4.160578911095796
  train_avg_answer_length: 73.33195497358143
  dev_samples: 948
  dev_avg_docs_per_question: 4.836497890295359
  dev_avg_answer_length: 57.857594936708864
  valid_samples: 948
  valid_avg_docs_per_question: 4.836497890295359


In [4]:
# Examine sample
sample = df_train.iloc[0]
print("Sample Question:")
print(sample['question'])
print("\nSample Answer (long-form):")
print(sample['answer'])
print(f"\nAnswer length: {len(sample['answer'].split())} words")
print(f"Number of context documents: {len(sample['docs'])}")

Sample Question:
When does the new bunk'd come out?

Sample Answer (long-form):
The new bunk'd episode 41 comes out on April 21, 2017, episode 42 comes out on April 28, 2017 and episode 42 is due to come out on May 24, 2017. 

Answer length: 31 words
Number of context documents: 4


## 3. Configure RAG System for Long-Form Generation

In [5]:
# Configure for ASQA (long-form answers)
config = RAGConfig(
    # Models
    encoder_model="sentence-transformers/all-MiniLM-L6-v2",
    generator_model="google/flan-t5-base",   # flan-t5-base: 3x smaller than large, fits in RAM

    # Paths
    models_dir="../models",
    output_dir="./rag_output_asqa",
    train_data_path=train_path,
    valid_data_path=dev_path,

    # Retriever training
    retriever_epochs=5,
    retriever_batch_size=16,
    retriever_lr=2e-5,
    retriever_patience=3,

    # Generator training
    generator_epochs=3,
    generator_batch_size=4,               # Increased from 2 (smaller model allows larger batch)
    generator_lr=5e-5,
    generator_max_input_tokens=512,
    generator_max_target_tokens=128,      # ASQA avg answer = 73 words; 128 tokens is sufficient

    # Checkpoint settings — sub-epoch saving so a crash loses at most N steps
    retriever_checkpoint_steps=100,       # ~90 sec between retriever checkpoints
    generator_checkpoint_steps=200,       # ~45 min between generator checkpoints
    checkpoint_dir=Path("./rag_output_asqa/checkpoints"),

    # Retrieval
    top_k=5,

    # Generation (inference only)
    max_new_tokens=256,

    # Device
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Configuration:")
print(f"  Encoder: {config.encoder_model}")
print(f"  Generator: {config.generator_model}")
print(f"  Max target tokens: {config.generator_max_target_tokens}")
print(f"  Generator batch size: {config.generator_batch_size}")
print(f"  Checkpoint dir: {config.checkpoint_dir}")
print(f"  Device: {config.device}")

Configuration:
  Encoder: sentence-transformers/all-MiniLM-L6-v2
  Generator: google/flan-t5-base
  Max target tokens: 128
  Generator batch size: 4
  Checkpoint dir: rag_output_asqa\checkpoints
  Device: cpu


## 4. Initialize RAG System

In [6]:
# Initialize RAG system
rag = RAGSystem(config=config)

# Use ASQA loader instead of default HotpotQA loader
rag.data_loader = loader


Initializing RAG System
Device: cpu
Encoder: sentence-transformers/all-MiniLM-L6-v2
Generator: google/flan-t5-base
Models directory: ..\models
Output directory: rag_output_asqa
Model cache directory: c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\notebooks\..\models

Loading models...
Loaded from cache: sentence-transformers--all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9787.12it/s]
BertModel LOAD REPORT from: ..\models\sentence-transformers--all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded from cache: google--flan-t5-base


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3510.54it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Models loaded successfully!



## 5. Train Retriever Model

In [7]:
# Create retriever training examples
retriever_examples = loader.create_retriever_examples()
print(f"Created {len(retriever_examples)} retriever training examples")

Creating retriever training examples...


Processing: 100%|██████████| 4353/4353 [00:00<00:00, 32409.00it/s]

Created 3450 retriever training examples
Created 3450 retriever training examples


In [8]:
# Train retriever
# Re-running this cell after a crash will auto-resume from the last checkpoint.
print("Training retriever on ASQA dataset...")
rag.train_retriever(
    epochs=config.retriever_epochs,
    batch_size=config.retriever_batch_size,
    lr=config.retriever_lr,
    resume_from_checkpoint=True,
)

Training retriever on ASQA dataset...

Training Retriever
Creating retriever training examples...


Processing: 100%|██████████| 4353/4353 [00:00<00:00, 33680.83it/s]
c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\notebooks\..\src\training\retriever_trainer.py:304: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler(_AMP_DEVICE, enabled=use_fp16)


Created 3450 retriever training examples
Retriever trainer initialized on cpu
Created DataLoader: 3450 examples, batch_size=16
Resuming retriever from checkpoint: rag_output_asqa\checkpoints\retriever_checkpoint.pt


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5363.56it/s]


   Resumed at epoch=6, step=0, best_loss=0.0021

Starting retriever training...
   Epochs: 5
   Learning rate: 2e-05
   Temperature: 20.0
   Accumulation steps: 2
   Mixed precision: True
   Checkpoint every 100 steps → rag_output_asqa\checkpoints

Training complete! Best loss: 0.0021
Model saved to: ..\models\retriever_trained



## 6. Evaluate Retriever

In [9]:
# Evaluate retriever on dev set
print("Evaluating retriever...")
retriever_metrics = rag.evaluate_retriever()

print("\nRetriever Metrics:")
for metric, value in retriever_metrics.items():
    print(f"  {metric}: {value:.4f}")

Evaluating retriever...

Evaluating Retriever

Evaluating retriever on 948 questions...
   Top-K values: [1, 3, 5]
Building corpus from validation set...
   Corpus size: 2010 unique documents
Loading corpus embeddings from cache: rag_output_asqa\corpus_embeds.pt

Evaluating retrieval performance...


Evaluating:   0%|          | 0/948 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 948/948 [00:05<00:00, 162.26it/s]


Retriever Evaluation Results:
   recall@1            : 0.6308
   precision@1         : 0.6308
   recall@3            : 0.7932
   precision@3         : 0.3688
   recall@5            : 0.8249
   precision@5         : 0.2458
   mrr                 : 0.7132


Retriever Metrics:
  recall@1: 0.6308
  precision@1: 0.6308
  recall@3: 0.7932
  precision@3: 0.3688
  recall@5: 0.8249
  precision@5: 0.2458
  mrr: 0.7132


## 7. Build FAISS Index

In [10]:
# Build corpus and index
print("Building corpus and FAISS index...")
rag.build_index()

Building corpus and FAISS index...

Building FAISS Index
Built corpus: 6269 unique passages from 4353 questions
Building FAISS index from 6269 documents...


Encoding: 100%|██████████| 98/98 [00:44<00:00,  2.22it/s]

Built FAISS index: 6269 documents, 384 dimensions
Saved index to rag_output_asqa\index
   - index.faiss: rag_output_asqa\index\index.faiss
   - meta.json: rag_output_asqa\index\meta.json



## 8. Train Generator Model

In [11]:
import gc

# Free the retriever encoder from RAM before loading the generator.
# Both models cannot comfortably coexist on CPU with limited memory.
if hasattr(rag, "retriever_trainer") and rag.retriever_trainer is not None:
    del rag.retriever_trainer.model
    rag.retriever_trainer = None
gc.collect()
print("Freed retriever encoder memory — ready for generator training")

Freed retriever encoder memory — ready for generator training


In [12]:
# Preview available training size (train_generator creates examples internally)
total_available = len(loader.df_train)
print(f"Training set size: {total_available} examples available")
print("Will use up to 2000 examples for generator training")

Training set size: 4353 examples available
Will use up to 2000 examples for generator training


In [13]:
# Train generator
# Re-running this cell after a crash will auto-resume from the last checkpoint.
print("Training generator on ASQA dataset...")
print("Note: This may take several hours for long-form generation")

rag.train_generator(
    epochs=config.generator_epochs,
    batch_size=config.generator_batch_size,
    lr=config.generator_lr,
    max_examples=2000,            # Cap for memory safety
    resume_from_checkpoint=True,  # Auto-resume from last checkpoint on crash
)

Training generator on ASQA dataset...
Note: This may take several hours for long-form generation

Training Generator
Creating generator training examples...


Processing:  46%|████▌     | 2000/4353 [00:00<00:00, 56938.31it/s]

Created 2000 generator training examples
Generator trainer initialized on cpu
Created DataLoader: 2000 examples, batch_size=4
Resuming generator from checkpoint: rag_output_asqa\checkpoints\generator_checkpoint.pt



Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3324.52it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


   Resumed at epoch=4, step=0, global_step=1500

Starting generator training...
   Epochs: 3
   Learning rate: 5e-05
   Warmup ratio: 0.1
   Gradient accumulation: 1
   Max input tokens: 512
   Max target tokens: 128
   Gradient checkpointing: enabled
   Checkpoint every 200 steps → rag_output_asqa\checkpoints


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.03s/it]

Model and tokenizer saved to ..\models\generator_trained

Training complete!
Model saved to: ..\models\generator_trained



## 9. Baseline 1: Normal RAG Inference

In [14]:
# Test on a sample question
test_question = df_dev.iloc[0]['question']
print(f"Test Question: {test_question}")

answer, contexts = rag.answer(test_question, return_contexts=True)
print(f"\nGenerated Answer:\n{answer}")
print(f"\nRetrieved {len(contexts)} contexts")

Test Question: Who has the highest goals in world football?
QA Pipeline initialized on cpu

Generated Answer:
Josef Bican has scored the most football goals in their career in history. Out of all the active players, Cristiano Ronaldo scored the most goals in his career with over 780 official senior career goals. Ali Daei of Iran has scored the most men's international goals in his career. He is the only player to score over 100 goals in international football with 109 goals. Christine Sinclair is the world's all-time leader for international goals scored for men or women with 187 goals, and is one of the most-capped active international footballers with 300 caps.

Retrieved 5 contexts


In [15]:
# Run inference on full dev set
print("Running Normal RAG inference on dev set...")

normal_rag_results = []

for idx, row in tqdm(df_dev.iterrows(), total=len(df_dev), desc="Normal RAG"):
    question = row['question']
    gold_answer = row['answer']
    
    try:
        # Generate answer
        predicted_answer, contexts = rag.answer(question, return_contexts=True)
        
        normal_rag_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': predicted_answer,
            'contexts': contexts,
            'num_contexts': len(contexts)
        })
    except Exception as e:
        print(f"Error on question {idx}: {e}")
        normal_rag_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': f"ERROR: {str(e)}",
            'contexts': [],
            'num_contexts': 0
        })

# Save results
results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

df_normal_rag = pd.DataFrame(normal_rag_results)
df_normal_rag.to_csv(results_dir / "asqa_normal_rag_predictions.csv", index=False)
print(f"\nâœ… Saved Normal RAG results: {len(df_normal_rag)} predictions")

Running Normal RAG inference on dev set...


Normal RAG:   0%|          | 0/948 [00:00<?, ?it/s]

QA Pipeline initialized on cpu


Normal RAG:   0%|          | 1/948 [00:03<1:02:15,  3.94s/it]

QA Pipeline initialized on cpu


...

QA Pipeline initialized on cpu


Normal RAG:   2%|▏         | 15/948 [00:45<46:25,  2.99s/it]

QA Pipeline initialized on cpu


Normal RAG:   2%|▏         | 15/948 [00:47<48:56,  3.15s/it]


KeyboardInterrupt: 

## 10. Black-box Answer Scoring (vs Ground Truth)

In [16]:
# Initialize answer-correctness metric bundle (no API key needed for lexical metrics)
metric_bundle = AnswerMetricBundle()

# Initialize LLM answer scorer (requires GOOGLE_API_KEY for LLM-as-judge)
answer_filter = AnswerFilter(
    api_key=os.getenv("GOOGLE_API_KEY"),
    model_name="gemini-2.5-flash",
    threshold=6.0,
    max_concurrent=10,
)

print("Initialized black-box answer scoring pipeline")
print("  - AnswerMetricBundle: token_f1, rouge_l + RAGAS answer_correctness/similarity")
print("  - AnswerFilter: LLM-as-judge scoring against ground truth")

Initialized LLM filter pipeline


In [17]:
# Test answer scoring on a sample
test_question = df_dev.iloc[0]['question']
test_gold = df_dev.iloc[0]['answer']
test_answer, test_contexts = rag.answer(test_question, return_contexts=True)

print(f"Question: {test_question}")
print(f"Gold answer: {test_gold[:200]}...")
print(f"Generated answer: {test_answer[:200]}...")
print(f"Contexts used: {len(test_contexts)} (all passed through, no filtering)")

# Lexical metrics (no API key needed)
metrics = metric_bundle.compute([test_question], [test_answer], [test_gold])
print(f"\nLexical Metrics (vs ground truth):")
print(f"  token_f1:  {metrics[0]['token_f1']:.3f}")
print(f"  rouge_l:   {metrics[0]['rouge_l']:.3f}")

# LLM-as-judge scoring (requires GOOGLE_API_KEY)
score_result = answer_filter.score_single(
    question=test_question,
    answer=test_answer,
    ground_truth=test_gold,
)
print(f"\nLLM Judge Score (vs ground truth):")
print(f"  Correctness:   {score_result.correctness_score:.1f}")
print(f"  Similarity:    {score_result.similarity_score:.1f}")
print(f"  Completeness:  {score_result.completeness_score:.1f}")
print(f"  Overall:       {score_result.overall_quality}")
print(f"  Reasoning:     {score_result.reasoning}")

QA Pipeline initialized on cpu
Question: Who has the highest goals in world football?

Original contexts: 5
Filtered contexts: 4

Context 1: Score=10.0, Relevant=True
  Reasoning: The context is highly relevant and helpful, providing the top goal scorers across different categories (all-time, active, men's international, overall international) which directly answers the question's most likely interpretation.

Context 2: Score=10.0, Relevant=True
  Reasoning: The context is highly relevant and directly provides specific answers for different interpretations of "highest goals" (all-time, active, men's international, and overall international).

Context 3: Score=10.0, Relevant=True
  Reasoning: The context directly answers who holds the record for the most goals in various categories (all-time career, active career, men's international, and overall international), making it highly relevant and helpful for the ambiguous question.

Context 4: Score=9.0, Relevant=True
  Reasoning: The contex

In [18]:
# Run black-box answer scoring on dev set
# Pipeline: retrieve all contexts -> generate -> score answer vs ground truth
print("Running black-box answer scoring on dev set...")

scored_results = []

for idx, row in tqdm(df_dev.iterrows(), total=len(df_dev), desc="Answer scoring"):
    question = row['question']
    gold_answer = row['answer']

    try:
        predicted_answer = rag.answer(question)

        scores = metric_bundle.compute([question], [predicted_answer], [gold_answer])[0]

        scored_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': predicted_answer,
            'token_f1': scores['token_f1'],
            'rouge_l': scores['rouge_l'],
            'answer_correctness': scores.get('answer_correctness', 0.0),
            'answer_similarity': scores.get('answer_similarity', 0.0),
        })
    except Exception as e:
        print(f"Error on question {idx}: {e}")
        scored_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': f"ERROR: {str(e)}",
            'token_f1': 0.0,
            'rouge_l': 0.0,
            'answer_correctness': 0.0,
            'answer_similarity': 0.0,
        })

df_scored = pd.DataFrame(scored_results)
df_scored.to_csv(results_dir / "asqa_scored_predictions.csv", index=False)
print(f"\nSaved scored results: {len(df_scored)} predictions")
print(f"Mean token_f1: {df_scored['token_f1'].mean():.4f}")
print(f"Mean rouge_l:  {df_scored['rouge_l'].mean():.4f}")


## 11. Quick Comparison

In [ ]:
# Compare Normal RAG vs Scored RAG
print("=" * 60)
print("Normal RAG vs Black-Box Scored RAG")
print("=" * 60)

print(f"\nNormal RAG predictions: {len(df_normal_rag)}")
print(f"Scored RAG predictions: {len(df_scored)}")

print(f"\nAnswer Correctness Metrics (Scored RAG):")
print(f"  Mean token_f1:           {df_scored['token_f1'].mean():.4f}")
print(f"  Mean rouge_l:            {df_scored['rouge_l'].mean():.4f}")
print(f"  Mean answer_correctness: {df_scored['answer_correctness'].mean():.4f}")
print(f"  Mean answer_similarity:  {df_scored['answer_similarity'].mean():.4f}")


## 12. Sample Outputs Comparison

In [ ]:
# Compare a few examples side by side
N_SHOW = 3

for i in range(min(N_SHOW, len(df_scored))):
    print(f"\n{"=" * 60}")
    print(f"Question: {df_scored.iloc[i]['question']}")
    print(f"Gold: {df_scored.iloc[i]['gold_answer'][:200]}...")
    print(f"\n[Normal RAG]")
    print(f"  {df_normal_rag.iloc[i]['predicted_answer'][:200]}...")
    print(f"\n[Scored RAG]")
    print(f"  {df_scored.iloc[i]['predicted_answer'][:200]}...")
    print(f"  token_f1={df_scored.iloc[i]['token_f1']:.3f}, "
          f"rouge_l={df_scored.iloc[i]['rouge_l']:.3f}")


## 13. Answer Reward Filter (Generate-Score-Retry)

The `AnswerRewardFilter` generates an answer from all retrieved contexts,
scores it against ground truth, and retries if the score is below threshold.
This is a pure black-box approach: no context filtering, just answer quality.


In [ ]:
# Build the generation function wrapper

def generation_fn(question: str, contexts: list[str]) -> str:
    """Generate an answer given a question and a list of context strings."""
    context_str = "\n\n".join(contexts)
    qa_pipeline = rag.create_qa_pipeline()
    prompt = f"Answer the question based on the context.\n\nContext: {context_str}\n\nQuestion: {question}\nAnswer:"
    return qa_pipeline.generate(prompt)

# Build the AnswerRewardFilter
reward_filter = AnswerRewardFilter(
    generation_fn=generation_fn,
    correctness_threshold=0.50,
    max_retries=3,
    dataset_type="asqa",
)

print("AnswerRewardFilter built.")
print(f"  Threshold : {reward_filter.correctness_threshold}")
print(f"  Max retries: {reward_filter.max_retries}")
print(f"  Weights   : {reward_filter.reward_computer.weights}")


In [ ]:
# Weight fitting on a calibration split
# Uses ground-truth token_f1 as the downstream signal

N_CAL = min(100, len(df_dev))
cal_df = df_dev.iloc[:N_CAL]
cal_questions = cal_df['question'].tolist()
cal_golds = cal_df['answer'].tolist()

# Generate baseline answers
cal_answers = [rag.answer(q) for q in tqdm(cal_questions, desc='Generating')]

# Compute metrics
cal_metrics = metric_bundle.compute(cal_questions, cal_answers, cal_golds)
downstream_y = [m['token_f1'] for m in cal_metrics]

# Fit weights
fitter = WeightFitter()
fitted_weights = fitter.fit(cal_metrics, downstream_y, method='correlation')
print("Fitted weights (correlation):")
for k, v in sorted(fitted_weights.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:.4f}")

# Update the weight bank
reward_filter.reward_computer.weight_bank.update('asqa', fitted_weights)
print("\nWeightBank updated with fitted weights.")


In [ ]:
# Run AnswerRewardFilter on eval set (generate -> score -> retry)
N_EVAL = min(50, len(df_dev))
eval_df = df_dev.iloc[:N_EVAL].copy()
eval_questions = eval_df['question'].tolist()
eval_golds = eval_df['answer'].tolist()
eval_passages = [
    row['docs'] if isinstance(row['docs'], list) else [row['docs']]
    for _, row in eval_df.iterrows()
]

# Run batch
rf_outputs = reward_filter.answer_batch(
    eval_questions, eval_passages, eval_golds, show_progress=True
)

rf_answers = [o[0] for o in rf_outputs]
rf_diagnostics = [o[1] for o in rf_outputs]

# Build results dataframe
rf_rows = []
for ans, diag in zip(rf_answers, rf_diagnostics):
    rf_rows.append({
        "question": diag.question,
        "ground_truth": diag.ground_truth,
        "generated_answer": ans,
        "correctness_score": diag.correctness_score,
        "accepted": diag.accepted,
        "retries": diag.retries,
        "answer_correctness": diag.reward.answer_correctness,
        "answer_similarity": diag.reward.answer_similarity,
        "token_f1": diag.reward.token_f1,
        "rouge_l": diag.reward.rouge_l,
        "composite": diag.reward.composite,
    })

df_rf = pd.DataFrame(rf_rows)
print(f"AnswerRewardFilter results: {len(df_rf)} samples")
print(f"  Accepted: {df_rf['accepted'].sum()} / {len(df_rf)} ({df_rf['accepted'].mean()*100:.1f}%)")
print(f"  Mean composite:  {df_rf['composite'].mean():.4f}")
print(f"  Mean token_f1:   {df_rf['token_f1'].mean():.4f}")
print(f"  Mean rouge_l:    {df_rf['rouge_l'].mean():.4f}")


In [ ]:
# Diagnostics summary
summary = AnswerRewardFilter.summarise_diagnostics(rf_diagnostics)
print("AnswerRewardFilter Batch Summary")
print("=" * 50)
for k, v in summary.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Show a few examples
N_SHOW = 3
for i in range(min(N_SHOW, len(rf_diagnostics))):
    d = rf_diagnostics[i]
    print(f"\n[Sample {i+1}]")
    print(f"  Q: {d.question[:100]}")
    print(f"  A: {d.generated_answer[:150]}")
    print(f"  Composite: {d.reward.composite:.3f}  Accepted: {d.accepted}  Retries: {d.retries}")
    print(f"  token_f1={d.reward.token_f1:.3f}, rouge_l={d.reward.rouge_l:.3f}")


In [ ]:
# Compare Normal RAG vs AnswerRewardFilter
import numpy as np

normal_scores = metric_bundle.compute(
    eval_questions,
    [rag.answer(q) for q in eval_questions],
    eval_golds,
)
normal_f1s = [s['token_f1'] for s in normal_scores]
rf_f1s = [d.reward.token_f1 for d in rf_diagnostics]

print(f"Normal RAG  mean token_f1: {np.mean(normal_f1s):.4f}")
print(f"Reward Filter mean token_f1: {np.mean(rf_f1s):.4f}")
print(f"Improvement: {(np.mean(rf_f1s) - np.mean(normal_f1s)):.4f}")
print(f"\n% retried: {np.mean([d.retries > 0 for d in rf_diagnostics])*100:.1f}%")
print(f"Avg retries: {np.mean([d.retries for d in rf_diagnostics]):.2f}")


## 14. Weight Analysis and Priors


In [ ]:
# Visualize answer-correctness weight priors
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 6))

datasets = list(ANSWER_WEIGHT_PRIORS.keys())
metrics_list = list(ANSWER_WEIGHT_PRIORS['universal'].keys())
x = np.arange(len(metrics_list))
width = 0.18

for i, ds in enumerate(datasets):
    weights = [ANSWER_WEIGHT_PRIORS[ds][m] for m in metrics_list]
    ax.bar(x + i * width, weights, width, label=ds)

ax.set_xlabel("Metric")
ax.set_ylabel("Weight")
ax.set_title("Answer-Correctness Weight Priors by Dataset")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_list, rotation=15)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# (Section cleared — old H-RRGF cell 41, no longer needed)


In [ ]:
# (Section cleared — old H-RRGF cell 42, no longer needed)


In [ ]:
# (Section cleared — old H-RRGF cell 43, no longer needed)


In [ ]:
# (Section cleared — old H-RRGF cell 44, no longer needed)


In [ ]:
# (Section cleared — old H-RRGF cell 45, no longer needed)


## Summary

This notebook evaluated a black-box RAG pipeline on ASQA:

1. **Normal RAG** - Standard retrieve-and-generate baseline
2. **Black-box Answer Scoring** - Score generated answers against ground truth using
   `AnswerMetricBundle` (token_f1, rouge_l, answer_correctness, answer_similarity)
3. **Answer Reward Filter** - Generate-then-score with retry loop using
   `AnswerRewardFilter` to improve answers

Key design: **no context filtering**. All retrieved passages go to the generator.
The only signal is whether the final answer matches the ground truth.
